In [ ]:
import pandas as pd

In [ ]:
# read in data file as pandas dataframe
df = pd.read_csv('C:/Users/Simon/Documents/diminutives.csv')
print(df.head())
print(df.dtypes)
print(len(df))

In [ ]:
#make a tinycrate
from tinycrate.tinycrate import TinyCrate, minimal_crate

In [ ]:
crate = minimal_crate()
print(crate.json())

In [ ]:
def make_examples(crate, row):
    ids = []
    head_id = "#" + str(index+1) + "-" + row["Data item"]
    example1_id = head_id + "-ex1"
    ids.append(example1_id)
    crate.add(
        "dmlex:Example",
        example1_id,
        {
            "dmlex:text": row["Example"] if pd.notnull(row["Example"]) else "",
            "dmlex:listingOrder": "1"
        }
    )
    if pd.notnull(row["Other citations"]):
        example2_id = head_id + "-ex2"
        ids.append(example2_id)
        crate.add(
            "dmlex:Example",
            example2_id,
            {
                "dmlex:text": row["Other citations"],
                "dmlex:listingOrder": "2"
            }
        )
    return ids

In [ ]:
def make_etymology(crate, row):
    head_id = "#" + str(index+1) + "-" + row["Data item"]
    etymol_id = head_id + "-etymol1"
    etymon_id = head_id + "-etymon1"
    note_ids = []
    note1_id = head_id + "-etyNote1"


    crate.add(
        "dmlex:Note",
        note1_id,
        {
            "dmlex:text": row["Phonol"] if pd.notnull(row["Phonol"]) else "",
            "dmlex:listingOrder": "1"
        }
    )
    note_ids = [note1_id]
    if pd.notnull(row["Alternative"]) and row["Alternative"] != "-":
        note2_id = head_id + "-etyNote2"
        crate.add(
            "dmlex:Note",
            note2_id,
            {
                "dmlex:text": row["Alternative"] if pd.notnull(row["Alternative"]) else "",
                "dmlex:listingOrder": "2"
            }
        )
        note_ids.append(note2_id)
    if row["Phonol"] and not row["Semantics"]:
        raise Error("Phonol but no Semantics")
    etymol_props = {
        "dmlex:etymonUnit": row["Semantics"] if pd.notnull(row["Semantics"]) else "",
        "dmlex:etymonType": "derivation",
        "dmlex:note": [{"@id": note_id} for note_id in note_ids],
        "dmlex:partOfSPeech": row["Gramm source" ] if pd.notnull(row["Gramm source"]) else ""
    }
                    
    crate.add(
        "dmlex:etymon",
        etymol_id,
        etymol_props
    )
    
    return [etymol_id]

In [ ]:
def make_definition(crate, row):
    ids = []
    head_id = "#" + str(index+1) + "-" + row["Data item"]
    def1_id = head_id + "-def1"
    ids.append(def1_id)
    crate.add(
        "dmlex:Definition",
        def1_id,
        {
            "dmlex:text": row["Source"] if pd.notnull(row["Source"]) else "",
        }
    )
    return ids

In [ ]:
crate = minimal_crate()
for index, row in df.iterrows():
    word = row["Data item"]
    head_id = "#" + str(index+1) + "-" + word
    sense_id = head_id + "-s1"
    entry_props = {"dmlex:headword": word, "dmlex:partOfSpeech": row["Part of speech"] if pd.notnull(row["Part of speech"]) else "", "dmlex:sense": {"@id":sense_id}}

#[{"@id": note_id} for note_id in note_ids]
    if pd.notnull(row["Sem classs"]):
        sense_props = {
            "dmlex:definition": [{"@id": definition} for definition in make_definition(crate, row)],
            "dmlex:etymology": [{"@id": etymology} for etymology in make_etymology(crate, row)],
            "dmlex:examples": [{"@id": example} for example in make_examples(crate, row)],
            "dmlex:label": row["Sem classs"]
    }
    else:
        sense_props = {
        "dmlex:definition": [{"@id": definition} for definition in make_definition(crate, row)],
        "dmlex:etymology": [{"@id": etymology} for etymology in make_etymology(crate, row)],
        "dmlex:examples": [{"@id": example} for example in make_examples(crate, row)],
    }
    #+1 for python purposes


    # notes
    # etymons
    # etymology
    # definitions
    # examples
    
    # senses
    # entries


    if pd.notnull(row["Geographical use"]):
        entry_props["spatialCoverage"] = row["Geographical use"]
    if pd.notnull(row["Comment"]):
        entry_props["dmlex:note"] = row["Comment"]
    crate.add("dmlex:Entry", head_id, entry_props)
    crate.add("dmlex:Sense", sense_id, sense_props)

    
#    sense_id = id + "-s1"

#    



In [ ]:
from pathlib import Path

In [ ]:
crate.write_json(Path("C:/Users/Simon/OneDrive/Desktop/diminutives/diminutives.json"))


In [ ]:
with open('C:/Users/Simon/OneDrive/Desktop/diminutives/diminutives-crate.json', 'w') as f:
    print(crate.json(), file = f)

    ####### old code follows
    if pd.notnull(row["Geographical use"]):
        geo_use = row["Geographical use"]
        props = {"dmlex:headword": word, "dmlex:PartOfSpeech": row["Part of speech"], "spatialCoverage": geo_use, "dmlex:Note": row["Comment"]}
    else:
        props = {"dmlex:headword": word, "dmlex:PartOfSpeech": row["Part of speech"]}
    crate.add("dmlex:Entry", id, props)
    
    if pd.notnull(row["Other citations"]):
        example2 = row["Other citations"]
    example1 = row["Example"]
    example1_id = id + "-e1"
    crate.add("dmlex:Example", example1_id,  {
        "dmlex:text": example1, "dmlex:listingOrder": "1"
    })
    if example2 != "":
        example2_id = id + "-e2"
    crate.add("dmlex:Example", example2_id, {
        "dmlex:text": example2, "dmlex:listingOrder": "2"
    })
 
    if example2:
        etymon_entity["dmlex:example"] = [{"@id1": example1_id, "@id2": example2_id}]
    elif example1:
        etymon_entity["dmlex:note"] = {"@id": example1_id}